### S1 ###

In [2]:
import pandas as pd
import numpy as np

# Read Data
file_path = r'.\results\normalized\S1 Quality.xlsx'
df = pd.read_excel(file_path)

# Fillup 
df[['Metric', 'Dataset']] = df[['Metric', 'Dataset']].ffill()

algorithm_columns = ['K-Means', 'Fairlet', 'BFKM', 'CCF', 'PP-NFP', 'PP-Gini', 'Rawlsian']

# Min-Max
for index, row in df.iterrows():
    metric_label = str(row['Metric'])
    values = pd.to_numeric(row[algorithm_columns], errors='coerce').values

    # identify ↓ or keyword
    is_lower_better = '↓' in metric_label or any(k in metric_label.upper() for k in ['SPD', 'FAIRNESS', 'ERROR'])

    if is_lower_better:
        best_val = np.nanmin(values)
        if best_val == 0:
            scaled_values = np.where(values == 0, 1.0, 0.0)
        else:
            scaled_values = best_val / values
    else:
        best_val = np.nanmax(values)
        if best_val == 0 or np.isnan(best_val):
            scaled_values = values
        else:
            scaled_values = values / best_val
    df.loc[index, algorithm_columns] = np.round(scaled_values, 6)

# Result
df.to_excel('S1_quality_Scaling.xlsx', index=False)
print("Processing complete!")

Processing complete!


In [3]:
import pandas as pd
import numpy as np

# Read Data
file_path = r'.\results\normalized\S1 Fairness.xlsx'
df = pd.read_excel(file_path)

# Fill up
df[['Metric', 'Dataset']] = df[['Metric', 'Dataset']].ffill()

algorithm_columns = ['K-Means', 'Fairlet', 'BFKM', 'CCF', 'PP-NFP', 'PP-Gini', 'Rawlsian']

# Min-Max
for index, row in df.iterrows():
    metric_label = str(row['Metric'])
    values = pd.to_numeric(row[algorithm_columns], errors='coerce').values
    is_lower_better = '↓' in metric_label or any(k in metric_label.upper() for k in ['SPD', 'FAIRNESS', 'ERROR'])

    if is_lower_better:
        best_val = np.nanmin(values)
        if best_val == 0:
            scaled_values = np.where(values == 0, 1.0, 0.0)
        else:
            scaled_values = best_val / values
    else:
        best_val = np.nanmax(values)
        if best_val == 0 or np.isnan(best_val):
            scaled_values = values
        else:
            scaled_values = values / best_val

    df.loc[index, algorithm_columns] = np.round(scaled_values, 6)

# Result
df.to_excel('S1_fair_Scaling.xlsx', index=False)
print("Processing complete!")

Processing complete!


### S2 ###

In [8]:
import pandas as pd
import numpy as np
import re

def scale_complex_table(input_path, output_path):
    # Read Data
    df = pd.read_excel(input_path)
    
    # Fill up
    df[['Metric', 'Dataset']] = df[['Metric', 'Dataset']].ffill()
    
    # Algorithm column name
    algo_cols = ['K-Means', 'Fairlet', 'BFKM', 'CCF', 'PP-NFP', 'PP-Gini', 'Rawlsian']
    
    def normalize_row(row):
        metric_label = str(row['Metric'])
        raw_vals = row[algo_cols].values
        numeric_vals = []
        
        for v in raw_vals:
            # if including k=, will be remove
            if isinstance(v, str) and 'k=' in v:
                return row[algo_cols]
            try:
                numeric_vals.append(float(v))
            except:
                numeric_vals.append(np.nan)
        
        vals_array = np.array(numeric_vals)
        
        is_lower_better = '↓' in metric_label or any(k in metric_label.upper() for k in ['SPD', 'FAIRNESS', 'ERROR', 'DB'])

        if is_lower_better:
            best_val = np.nanmin(vals_array)
            if best_val == 0:
                scaled = np.where(vals_array == 0, 1.0, 0.0)
            else:
                scaled = best_val / vals_array
        else:
            best_val = np.nanmax(vals_array)
            if best_val == 0 or np.isnan(best_val):
                scaled = vals_array
            else:
                scaled = vals_array / best_val
        
        return pd.Series([f"{x:.6f}" if not np.isnan(x) else np.nan for x in scaled], index=algo_cols)

    for index, row in df.iterrows():
        try:
            val_check = str(row[algo_cols[0]])
            if 'k=' in val_check:
                scaled_row = normalize_row(row)
                df.loc[index, algo_cols] = scaled_row
        except:
            continue

    # Result
    df.to_excel(output_path, index=False)
    print(f"Processing complete！")

# Execute
scale_complex_table( r'.\results\normalized\S2 Quality.xlsx', 'Scaled_S2_q_Table.xlsx')

Processing complete！


In [14]:
import pandas as pd
import numpy as np
import re

def scale_complex_table(input_path, output_path):
    # Read Data
    df = pd.read_excel(input_path)
    
    # Fill up
    df[['Metric', 'Dataset']] = df[['Metric', 'Dataset']].ffill()
    
    # Algorithm column name
    algo_cols = ['K-Means', 'Fairlet', 'BFKM', 'CCF', 'PP-NFP', 'PP-Gini', 'Rawlsian']
    
    def normalize_row(row):
        metric_label = str(row['Metric'])
        raw_vals = row[algo_cols].values
        numeric_vals = []
        
        for v in raw_vals:
            if isinstance(v, str) and 'k=' in v:
                return row[algo_cols]
            try:
                numeric_vals.append(float(v))
            except:
                numeric_vals.append(np.nan)
        
        vals_array = np.array(numeric_vals)
        
        is_lower_better = '↓' in metric_label or any(k in metric_label.upper() for k in ['SPD', 'FAIRNESS', 'ERROR', 'DB'])

        if is_lower_better:
            best_val = np.nanmin(vals_array)
            if best_val == 0:
                scaled = np.where(vals_array == 0, 1.0, 0.0)
            else:
                scaled = best_val / vals_array
        else:
            best_val = np.nanmax(vals_array)
            if best_val == 0 or np.isnan(best_val):
                scaled = vals_array
            else:
                scaled = vals_array / best_val
        
        return pd.Series([f"{x:.6f}" if not np.isnan(x) else np.nan for x in scaled], index=algo_cols)

    for index, row in df.iterrows():
        try:
            val_check = str(row[algo_cols[0]])
            if 'k=' in val_check:
                scaled_row = normalize_row(row)
                df.loc[index, algo_cols] = scaled_row
        except:
            continue

    # Result
    df.to_excel(output_path, index=False)
    print(f"Processing complete!")

# Execute
scale_complex_table( r'.\results\normalized\S2 Fairness.xlsx', 'Scaled_S2_f_Table.xlsx')

Processing complete!
